# Figure 7 — Trace-element comparisons


## 1. Imports and shared 00_Setup

This notebook compares the measured trace-element and major-element data directly. It uses Arial, 8-point ordinary text, 9-point bold uppercase panel labels, full axes boxes, light gridlines, 300 dpi PNG export, and editable PDF/SVG text. The code is linear and contains no custom functions. Mixing lines, mixing percentages, calculated endmembers, and the model legend are intentionally not included.


In [1]:
from pathlib import Path
import hashlib
import re

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator, FormatStrFormatter
import numpy as np
import pandas as pd
from PIL import Image

SINGLE_COLUMN_WIDTH_IN = 3.55
DOUBLE_COLUMN_WIDTH_IN = 7.25
MAXIMUM_FIGURE_HEIGHT_IN = 7.8
FIGURE_WIDTH_IN = DOUBLE_COLUMN_WIDTH_IN
FIGURE_HEIGHT_IN = 6.2
BASE_TEXT_SIZE_PT = 8.0
AXIS_LABEL_SIZE_PT = 8.0
PANEL_LABEL_SIZE_PT = 9.0
LEGEND_SIZE_PT = 6.0
MINIMUM_LINE_WIDTH_PT = 0.28
EXPORT_DPI = 300
PUBLICATION_FONT = "Arial"

installed_font_names = {entry.name for entry in font_manager.fontManager.ttflist}
if PUBLICATION_FONT not in installed_font_names:
    raise RuntimeError("Arial is required by 00_Setup.")
mpl.rcParams.update({
    "font.family":"sans-serif", "font.sans-serif":[PUBLICATION_FONT],
    "font.size":BASE_TEXT_SIZE_PT, "axes.labelsize":AXIS_LABEL_SIZE_PT,
    "axes.titlesize":BASE_TEXT_SIZE_PT, "xtick.labelsize":BASE_TEXT_SIZE_PT,
    "ytick.labelsize":BASE_TEXT_SIZE_PT, "legend.fontsize":LEGEND_SIZE_PT,
    "legend.title_fontsize":LEGEND_SIZE_PT, "figure.facecolor":"white",
    "axes.facecolor":"white", "axes.spines.top":True, "axes.spines.right":True,
    "axes.linewidth":0.5, "lines.linewidth":0.75,
    "xtick.direction":"in", "ytick.direction":"in",
    "xtick.major.width":0.5, "ytick.major.width":0.5,
    "savefig.dpi":EXPORT_DPI, "savefig.facecolor":"white",
    "savefig.transparent":False, "pdf.fonttype":42, "ps.fonttype":42,
    "svg.fonttype":"none",
})


## 2. Paths and public S1–S3 workbooks

Figure 7 reads only the public S1 and S3 sheets needed for the plotted populations. S2 is checked as part of the shared package but is not used in the plotted data.


In [2]:
PROJECT_ROOT_OVERRIDE = None
PROJECT_ROOT = PROJECT_ROOT_OVERRIDE
if PROJECT_ROOT is None:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "SA_MI_Draft_FinalFigures_Work" / "DataSheets").is_dir():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the project or set PROJECT_ROOT_OVERRIDE.")
PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
WORK_ROOT = PROJECT_ROOT / "SA_MI_Draft_FinalFigures_Work"
DATA_DIRECTORY = WORK_ROOT / "DataSheets" / "Updated_Oct20"
FIGURE_DIRECTORY = WORK_ROOT / "Nov6_AllFigures" / "Figure_07_TE_Plot_Main"
OUTPUT_DIRECTORY = FIGURE_DIRECTORY / "Figure_07_TE_Plot_Main_Final"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

S1_PATH = Path('/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/GP_Combined/SEIMS_SEM_Raman_LAICPMS/Combining_All_Dechert_Data/Combining_All_Dechert_Data_S1_20261110.xlsx')
S2_PATH = DATA_DIRECTORY / "Dechert_S2_Standards.xlsx"
S3_PATH = DATA_DIRECTORY / "Dechert_S3_Compilations.xlsx"
for workbook_path in [S1_PATH,S2_PATH,S3_PATH]:
    if not workbook_path.is_file():
        raise FileNotFoundError(f"Missing supplemental workbook: {workbook_path}")
SOURCE_HASHES_BEFORE = {
    "S1":hashlib.sha256(S1_PATH.read_bytes()).hexdigest(),
    "S2":hashlib.sha256(S2_PATH.read_bytes()).hexdigest(),
    "S3":hashlib.sha256(S3_PATH.read_bytes()).hexdigest(),
}


## 3. Read the source sheets and prepare the comparison tables


In [3]:
SAMPLE_COLORS = {"74D":"#F4A261","74A":"#D55E00","75":"#56B4E9","76A":"#0072B2"}
SAMPLE_LABELS = {"74D":"B Middle","74A":"B Base","75":"G Top","76A":"G Middle"}
MINGLED_SAMPLE_COLORS = {"B light":"#EE3377","B dark":"#808080","G light":"#66C2C2","G dark":"#000000"}
KUEHN_COLORS = {"B":"#D4A017","G":"#006D77"}
KUEHN_MARKER_SIZE = 64.0
B_DACITE_COLOR = "#9E2A4F"
MATRIX_GLASS_MARKER = "x"
MI_TRACE_COLUMNS = {"Sr":"88Sr_TE_MI_mean","Rb":"85Rb_TE_MI_mean","Ba":"137Ba_TE_MI_mean"}
MI_TRACE_STD_COLUMNS = {"Sr":"88Sr_TE_MI_std_1sigma","Rb":"85Rb_TE_MI_std_1sigma","Ba":"137Ba_TE_MI_std_1sigma"}
MI_K2O_SOURCE_COLUMN = "K2O_SEMGlass_AnhydoNorm"
MI_SIO2_SOURCE_COLUMN = "SiO2_SEMGlass_AnhydoNorm"
PHASE_MARKERS = {"plag":"o","pxy":"s"}

mi_source = pd.read_excel(S1_PATH, sheet_name="MI_Hosts_Plag_Pxy")
vein_source = pd.read_excel(S1_PATH, sheet_name="Dacite_Vein")
whole_rock_source = pd.read_excel(S1_PATH, sheet_name="Whole_Rock")
kuehn_source = pd.read_excel(S3_PATH, sheet_name="Kuehn2009")
mingled_matrix_source = pd.read_excel(S1_PATH, sheet_name="Mingled_Matrix_Glass")

main_view = mi_source.copy()
sem_columns = [column for column in main_view.columns if column.endswith("_SEMGlass_AnhydoNorm")]
main_view.loc[~main_view["Plot_Eligible_SEM_Glass"].eq(True), sem_columns] = np.nan

vein_view = vein_source.copy().rename(columns={column:column.removesuffix("_TE_ppm") for column in vein_source.columns if column.endswith("_TE_ppm")})
for oxide in ["SiO2","TiO2","Al2O3","FeOt","MnO","MgO","CaO","Na2O","K2O","P2O5"]:
    vein_view[oxide] = vein_source[f"{oxide}_SEMGlass_AnhydoNorm"]
vein_view["TE QC Comments"] = "Curated S1 TE record; original comment not distributed"

icpms_view = whole_rock_source.rename(columns={"Sample":"Sample ID", **{column:column.removesuffix("_ICPMS_ppm")+" ppm" for column in whole_rock_source.columns if column.endswith("_ICPMS_ppm")}}).copy()
xrf_view = pd.DataFrame(np.nan,index=range(65),columns=range(5),dtype=object)
xrf_view.iloc[0,1:5] = whole_rock_source["Sample"].to_numpy()
xrf_view.iloc[19,1] = "Normalized Major Elements"
for index, oxide in enumerate(["SiO2","TiO2","Al2O3","FeOt","MgO","CaO","Na2O","K2O","P2O5","MnO"]):
    xrf_view.iloc[20+index,0] = oxide
    xrf_view.iloc[20+index,1:5] = whole_rock_source[f"{oxide}_XRF_Anhydrous_wt_pct"].to_numpy()
xrf_view.iloc[32,1] = "Trace Elements"
for index,column in enumerate(column for column in whole_rock_source.columns if column.endswith("_XRF_ppm")):
    xrf_view.iloc[33+index,0] = column.removesuffix("_XRF_ppm")
    xrf_view.iloc[33+index,1:5] = whole_rock_source[column].to_numpy()

mi_work = main_view.copy()
mi_work["sample_key"] = pd.NA
for sample_key in SAMPLE_COLORS:
    mi_work.loc[mi_work["base_name"].astype("string").str.startswith(sample_key,na=False),"sample_key"] = sample_key
mi_work["mineral_phase"] = mi_work["mineral_phase"].astype("string").str.strip().str.casefold().replace({"opx":"pxy"})
mi_work["is_matrix_glass"] = (
    mi_work["base_name"].astype("string").str.casefold().str.contains(r"(?:^|[_\s-])mtx(?:$|[_\s-])|matrix glass",regex=True,na=False)
    | mi_work["mineral_phase"].isin(["mtx","matrix","matrix glass","groundmass glass"])
)
mi_work["discard_sims_yes"] = mi_work["Discard_SIMS"].fillna("").astype(str).str.strip().str.casefold().eq("yes")
mi_base_mask = ~mi_work["is_matrix_glass"] & ~mi_work["discard_sims_yes"] & mi_work["sample_key"].isin(SAMPLE_COLORS) & mi_work["mineral_phase"].isin(PHASE_MARKERS)
mi = mi_work.loc[mi_base_mask,["base_name","sample_key","mineral_phase","Discard_SIMS",MI_K2O_SOURCE_COLUMN,MI_SIO2_SOURCE_COLUMN,*MI_TRACE_COLUMNS.values(),*MI_TRACE_STD_COLUMNS.values()]].copy()
for element,column in MI_TRACE_COLUMNS.items():
    mi[element] = pd.to_numeric(mi[column],errors="coerce")
for element,column in MI_TRACE_STD_COLUMNS.items():
    mi[f"{element}_std_1sigma"] = pd.to_numeric(mi[column],errors="coerce")
mi["K2O"] = pd.to_numeric(mi[MI_K2O_SOURCE_COLUMN],errors="coerce")
mi["SiO2"] = pd.to_numeric(mi[MI_SIO2_SOURCE_COLUMN],errors="coerce")

main_matrix = mi_work.loc[
    mi_work["is_matrix_glass"]
    & mi_work["sample_key"].isin(SAMPLE_COLORS)
    & mi_work[MI_K2O_SOURCE_COLUMN].notna()
    & mi_work[MI_SIO2_SOURCE_COLUMN].notna()
].copy()
main_matrix["K2O"] = pd.to_numeric(main_matrix[MI_K2O_SOURCE_COLUMN],errors="coerce")
main_matrix["SiO2"] = pd.to_numeric(main_matrix[MI_SIO2_SOURCE_COLUMN],errors="coerce")

mingled_matrix = mingled_matrix_source.copy()
mingled_matrix["sample_key"] = pd.NA
mingled_matrix.loc[mingled_matrix["Dataset Description"].astype("string").str.casefold().eq("b middle") & mingled_matrix["Type of Pumice"].astype("string").str.casefold().eq("light"),"sample_key"] = "B light"
mingled_matrix.loc[mingled_matrix["Dataset Description"].astype("string").str.casefold().eq("b middle") & mingled_matrix["Type of Pumice"].astype("string").str.casefold().eq("dark"),"sample_key"] = "B dark"
mingled_matrix.loc[mingled_matrix["Dataset Description"].astype("string").str.casefold().eq("g middle") & mingled_matrix["Type of Pumice"].astype("string").str.casefold().eq("light"),"sample_key"] = "G light"
mingled_matrix.loc[mingled_matrix["Dataset Description"].astype("string").str.casefold().eq("g middle") & mingled_matrix["Type of Pumice"].astype("string").str.casefold().eq("dark"),"sample_key"] = "G dark"
mingled_matrix["K2O"] = pd.to_numeric(mingled_matrix["K2O_SEMGlass_AnhydoNorm"],errors="coerce")
mingled_matrix["SiO2"] = pd.to_numeric(mingled_matrix["SiO2_SEMGlass_AnhydoNorm"],errors="coerce")
mingled_matrix = mingled_matrix.loc[mingled_matrix["sample_key"].isin(MINGLED_SAMPLE_COLORS)].copy()

kuehn = kuehn_source.copy()
unit_text = kuehn["Analyzed Unit"].fillna("").astype(str) + " | " + kuehn["Sample ID"].fillna("").astype(str)
kuehn["Tephra"] = np.select([
    unit_text.str.contains(r"(?i)(?:\bbed\s*B(?:\b|[-\d])|(?:^|[/\s])B\d?\b)",regex=True),
    unit_text.str.contains(r"(?i)(?:\bbed\s*G(?:\b|-)|(?:^|[/\s])G\b)",regex=True),
],["B","G"],default=pd.NA)
kuehn = kuehn.loc[kuehn["Tephra"].isin(["B","G"])].copy()
for element in ["Sr","Rb","Ba","K2O","SiO2"]:
    kuehn[element] = pd.to_numeric(kuehn[element],errors="coerce")

dacite_all = vein_view.loc[vein_view["TE Sample"].notna()].copy()
dacite = dacite_all.rename(columns={"88Sr":"Sr","85Rb":"Rb","137Ba":"Ba"}).copy()
for element in ["Sr","Rb","Ba","K2O","SiO2"]:
    dacite[element] = pd.to_numeric(dacite[element],errors="coerce")

target_sample_ids = ["GP-74D","GP-74A","GP-75","GP-76A"]
whole_rock_icpms = icpms_view.loc[icpms_view["Sample ID"].isin(target_sample_ids)].copy().rename(columns={"Sr ppm":"Sr","Rb ppm":"Rb","Ba ppm":"Ba"})
whole_rock_icpms["sample_key"] = whole_rock_icpms["Sample ID"].str.replace("GP-","",regex=False)
for element in ["Sr","Rb","Ba"]:
    whole_rock_icpms[element] = pd.to_numeric(whole_rock_icpms[element],errors="coerce")
xrf_sample_columns = {str(xrf_view.iloc[0,column]).strip():column for column in range(1,xrf_view.shape[1]) if pd.notna(xrf_view.iloc[0,column])}
xrf_majors = pd.DataFrame({
    "Sample ID":target_sample_ids,
    "K2O":[pd.to_numeric(pd.Series([xrf_view.iloc[28,xrf_sample_columns[sample]]]),errors="coerce").iloc[0] for sample in target_sample_ids],
    "SiO2":[pd.to_numeric(pd.Series([xrf_view.iloc[20,xrf_sample_columns[sample]]]),errors="coerce").iloc[0] for sample in target_sample_ids],
})
whole_rock = whole_rock_icpms.merge(xrf_majors,on="Sample ID",how="left",validate="one_to_one")
assert len(whole_rock) == 4
for element in ["Sr","Rb","Ba","K2O","SiO2"]:
    assert whole_rock[element].notna().all()


## 4. Plot measured data only


In [4]:
PAIR_DEFINITIONS = [("A)","Sr","Rb"),("B)","Sr","Ba"),("C)","Sr","SiO2"),("D)","K2O","SiO2")]
AXIS_LABELS = {"Sr":"Sr (ppm)","Rb":"Rb (ppm)","Ba":"Ba (ppm)","K2O":"K$_2$O (wt%)","SiO2":"SiO$_2$ (wt%)"}
figure, axes = plt.subplots(2,2,figsize=(FIGURE_WIDTH_IN,FIGURE_HEIGHT_IN))
axes = np.asarray(axes).ravel()
figure.subplots_adjust(left=0.07,right=0.99,bottom=0.07,top=0.85,wspace=0.20,hspace=0.27)
for axis,(panel,x_element,y_element) in zip(axes,PAIR_DEFINITIONS):
    if panel == "D)":
        for mingled_key,color in MINGLED_SAMPLE_COLORS.items():
            mingled_subset = mingled_matrix.loc[mingled_matrix["sample_key"].eq(mingled_key)].dropna(subset=[x_element,y_element])
            if not mingled_subset.empty:
                axis.scatter(mingled_subset[x_element],mingled_subset[y_element],color=color,marker=MATRIX_GLASS_MARKER,s=36,alpha=0.58,linewidths=1.35,zorder=2)
    for sample_key,color in SAMPLE_COLORS.items():
        for phase,marker in PHASE_MARKERS.items():
            subset = mi.loc[mi["sample_key"].eq(sample_key) & mi["mineral_phase"].eq(phase)].dropna(subset=[x_element,y_element])
            if len(subset) == 0:
                continue
            xerr = subset[f"{x_element}_std_1sigma"] if f"{x_element}_std_1sigma" in subset.columns else None
            yerr = subset[f"{y_element}_std_1sigma"] if f"{y_element}_std_1sigma" in subset.columns else None
            if xerr is not None and not xerr.notna().any():
                xerr = None
            if yerr is not None and not yerr.notna().any():
                yerr = None
            if xerr is not None or yerr is not None:
                axis.errorbar(subset[x_element],subset[y_element],xerr=xerr,yerr=yerr,fmt="none",ecolor=color,elinewidth=0.55,capsize=1.5,alpha=0.48,zorder=2.4)
            axis.scatter(subset[x_element],subset[y_element],color=color,marker=marker,s=44 if phase=="plag" else 40,alpha=0.80,edgecolors="black",linewidths=0.28,zorder=3)
    if panel == "D)":
        for sample_key,color in SAMPLE_COLORS.items():
            main_matrix_subset = main_matrix.loc[main_matrix["sample_key"].eq(sample_key)].dropna(subset=[x_element,y_element])
            if not main_matrix_subset.empty:
                axis.scatter(main_matrix_subset[x_element],main_matrix_subset[y_element],color=color,marker=MATRIX_GLASS_MARKER,s=36,alpha=0.88,linewidths=1.35,zorder=4)
    for tephra,color in KUEHN_COLORS.items():
        subset = kuehn.loc[kuehn["Tephra"].eq(tephra)].dropna(subset=[x_element,y_element])
        axis.scatter(subset[x_element],subset[y_element],color=color,marker="+",s=KUEHN_MARKER_SIZE,linewidths=1.92,alpha=0.86,zorder=5)
    if panel != "D)":
        dacite_subset = dacite.dropna(subset=[x_element,y_element])
        axis.scatter(dacite_subset[x_element],dacite_subset[y_element],color=B_DACITE_COLOR,marker="D",s=47.6,alpha=0.88,edgecolors="black",linewidths=0.28,zorder=5)
        for sample_key,color in SAMPLE_COLORS.items():
            subset = whole_rock.loc[whole_rock["sample_key"].eq(sample_key)]
            axis.scatter(subset[x_element],subset[y_element],color=color,marker="*",s=58.4,alpha=0.98,edgecolors="black",linewidths=0.28,zorder=6)
        all_x = pd.concat([mi[x_element],kuehn[x_element],dacite[x_element],whole_rock[x_element]],ignore_index=True)
        all_y = pd.concat([mi[y_element],kuehn[y_element],dacite[y_element],whole_rock[y_element]],ignore_index=True)
    else:
        # Panel D focuses on melt inclusions and glasses; omit dacite-vein and whole-rock points.
        all_x = pd.concat([mi[x_element],kuehn[x_element]],ignore_index=True)
        all_y = pd.concat([mi[y_element],kuehn[y_element]],ignore_index=True)
    if panel == "D)":
        all_x = pd.concat([all_x,main_matrix[x_element],mingled_matrix[x_element]],ignore_index=True)
        all_y = pd.concat([all_y,main_matrix[y_element],mingled_matrix[y_element]],ignore_index=True)
    finite_x = pd.to_numeric(all_x,errors="coerce").dropna().to_numpy(float)
    finite_y = pd.to_numeric(all_y,errors="coerce").dropna().to_numpy(float)
    x_pad = max(0.07*(finite_x.max()-finite_x.min()),0.1)
    y_pad = max(0.08*(finite_y.max()-finite_y.min()),0.1)
    axis.set_xlim(max(0.0,float(finite_x.min()-x_pad)),float(finite_x.max()+x_pad))
    axis.set_ylim(max(0.0,float(finite_y.min()-y_pad)),float(finite_y.max()+y_pad))
    axis.set_xlabel(AXIS_LABELS[x_element])
    axis.set_ylabel(AXIS_LABELS[y_element])
    axis.set_title(f"{panel} {AXIS_LABELS[y_element].replace(' (ppm)','').replace(' (wt%)','')} vs. {AXIS_LABELS[x_element].replace(' (ppm)','').replace(' (wt%)','')}",loc="left",fontweight="bold",fontsize=PANEL_LABEL_SIZE_PT,pad=3)
    axis.grid(True,linestyle=":",linewidth=0.45,color="0.78",alpha=0.65)
    axis.set_axisbelow(True)
    axis.xaxis.set_major_locator(MaxNLocator(nbins=5))
    axis.yaxis.set_major_locator(MaxNLocator(nbins=5))
    if x_element == "K2O":
        axis.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    if y_element == "K2O":
        axis.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    for spine in axis.spines.values():
        spine.set_visible(True); spine.set_color("black"); spine.set_linewidth(0.5)

study_handles = [
    Patch(facecolor=SAMPLE_COLORS["74D"],edgecolor="none",label="B Middle"),
    Patch(facecolor=SAMPLE_COLORS["74A"],edgecolor="none",label="B Base"),
    Patch(facecolor=SAMPLE_COLORS["75"],edgecolor="none",label="G Top"),
    Patch(facecolor=SAMPLE_COLORS["76A"],edgecolor="none",label="G Middle"),
    Line2D([],[],marker="o",linestyle="none",markersize=5.0,markerfacecolor="0.45",markeredgecolor="black",label="Plagioclase-hosted MI"),
    Line2D([],[],marker="s",linestyle="none",markersize=5.0,markerfacecolor="0.45",markeredgecolor="black",label="Pyroxene-hosted MI"),
    Line2D([],[],marker=MATRIX_GLASS_MARKER,linestyle="none",markersize=np.sqrt(36),markerfacecolor="none",markeredgecolor="0.35",color="0.35",markeredgewidth=1.35,label="Matrix glass (main-type)"),
    Line2D([],[],marker="*",linestyle="none",markersize=np.sqrt(58.4),markerfacecolor="0.45",markeredgecolor="black",label="Whole-rock"),
]
figure.legend(handles=study_handles,title="Melt inclusions and whole-rock data",ncol=3,loc="upper center",bbox_to_anchor=(0.36,0.99),frameon=True,fancybox=False,framealpha=1.0,edgecolor="black",fontsize=6.0,title_fontsize=6.0,borderpad=0.4,labelspacing=0.25,columnspacing=0.9,handletextpad=0.5)
matrix_handles = [
    Line2D([],[],marker="+",linestyle="none",color=KUEHN_COLORS["B"],markersize=np.sqrt(KUEHN_MARKER_SIZE),markeredgewidth=1.92,label="B Kuehn"),
    Line2D([],[],marker="+",linestyle="none",color=KUEHN_COLORS["G"],markersize=np.sqrt(KUEHN_MARKER_SIZE),markeredgewidth=1.92,label="G Kuehn"),
    Line2D([],[],marker=MATRIX_GLASS_MARKER,linestyle="none",color=MINGLED_SAMPLE_COLORS["B light"],markersize=np.sqrt(36),markeredgewidth=1.35,label="B light mingled glass"),
    Line2D([],[],marker=MATRIX_GLASS_MARKER,linestyle="none",color=MINGLED_SAMPLE_COLORS["B dark"],markersize=np.sqrt(36),markeredgewidth=1.35,label="B dark mingled glass"),
    Line2D([],[],marker=MATRIX_GLASS_MARKER,linestyle="none",color=MINGLED_SAMPLE_COLORS["G light"],markersize=np.sqrt(36),markeredgewidth=1.35,label="G light mingled glass"),
    Line2D([],[],marker=MATRIX_GLASS_MARKER,linestyle="none",color=MINGLED_SAMPLE_COLORS["G dark"],markersize=np.sqrt(36),markeredgewidth=1.35,label="G dark mingled glass"),
    Line2D([],[],marker="D",linestyle="none",markerfacecolor=B_DACITE_COLOR,markeredgecolor="black",markersize=np.sqrt(47.6),label="B Trinity Dacite Vein"),
]
figure.legend(handles=matrix_handles,title="Matrix glasses",ncol=2,loc="upper center",bbox_to_anchor=(0.72,0.99),frameon=True,fancybox=False,framealpha=1.0,edgecolor="black",fontsize=6.0,title_fontsize=6.0,borderpad=0.4,labelspacing=0.25,columnspacing=0.8,handletextpad=0.5)
FIGURE_STEM = "Figure_07_TE_Plot_Main"
EXPORT_PATHS = {}
for output_format in ["pdf","png","svg"]:
    output_path = OUTPUT_DIRECTORY / f"{FIGURE_STEM}.{output_format}"
    save_options = {"format":output_format,"facecolor":"white","transparent":False}
    if output_format == "png":
        save_options["dpi"] = EXPORT_DPI
    figure.savefig(output_path,**save_options)
    if not output_path.is_file() or output_path.stat().st_size == 0:
        raise IOError(f"Expected export was not created: {output_path}")
    EXPORT_PATHS[output_format] = output_path
plt.close(figure)


## 5. Validate the direct comparison and exports


In [5]:
assert len(axes) == 4
assert [axis.get_title(loc="left").split()[0] for axis in axes] == ["A)","B)","C)","D)"]
assert [(axis.get_xlabel(),axis.get_ylabel()) for axis in axes] == [("Sr (ppm)","Rb (ppm)"),("Sr (ppm)","Ba (ppm)"),("Sr (ppm)","SiO$_2$ (wt%)"),("K$_2$O (wt%)","SiO$_2$ (wt%)")]
assert SAMPLE_COLORS == {"74D":"#F4A261","74A":"#D55E00","75":"#56B4E9","76A":"#0072B2"}
assert PHASE_MARKERS == {"plag":"o","pxy":"s"}
assert KUEHN_COLORS == {"B":"#D4A017","G":"#006D77"}
assert B_DACITE_COLOR == "#9E2A4F"
assert MATRIX_GLASS_MARKER == "x"
assert len(main_matrix) > 0
assert set(mingled_matrix["sample_key"]) == set(MINGLED_SAMPLE_COLORS)
assert len(whole_rock) == 4 and set(whole_rock["Sample ID"]) == set(target_sample_ids)
assert tuple(round(value,8) for value in figure.get_size_inches()) == (7.25,6.2)
assert all(spine.get_visible() for axis in axes for spine in axis.spines.values())
assert all(path.is_file() and path.stat().st_size > 0 for path in EXPORT_PATHS.values())
with Image.open(EXPORT_PATHS["png"]) as image:
    assert image.size == (2175,1860)
svg_text = EXPORT_PATHS["svg"].read_text(encoding="utf-8",errors="replace")
assert "<text" in svg_text and "<image" not in svg_text
assert b"/Type3" not in EXPORT_PATHS["pdf"].read_bytes()
SOURCE_HASHES_AFTER = {
    "S1":hashlib.sha256(S1_PATH.read_bytes()).hexdigest(),
    "S2":hashlib.sha256(S2_PATH.read_bytes()).hexdigest(),
    "S3":hashlib.sha256(S3_PATH.read_bytes()).hexdigest(),
}
assert SOURCE_HASHES_AFTER == SOURCE_HASHES_BEFORE
print("Figure 7 direct comparison complete.")
print("Mixing lines, mixing percentages, endmembers, and model legend were removed.")
print("Outputs:", {key:str(value) for key,value in EXPORT_PATHS.items()})


Figure 7 direct comparison complete.
Mixing lines, mixing percentages, endmembers, and model legend were removed.
Outputs: {'pdf': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_07_TE_Plot_Main/Figure_07_TE_Plot_Main_Final/Figure_07_TE_Plot_Main.pdf', 'png': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_07_TE_Plot_Main/Figure_07_TE_Plot_Main_Final/Figure_07_TE_Plot_Main.png', 'svg': '/Users/annikadechert/Library/Mobile Documents/com~apple~CloudDocs/!!!Cal_Postdoc/Berkeley Postdoc/Glacier Peak Efforts/SA_MI_Draft_FinalFigures_Work/Nov6_AllFigures/Figure_07_TE_Plot_Main/Figure_07_TE_Plot_Main_Final/Figure_07_TE_Plot_Main.svg'}
